# Piano CV Pipeline — Vision-Based Key-Press Detection

**Group B (Deployable System):** Runs on **raw video only** at inference time.
No provided hand skeleton JSON, no fingering labels, no annotations at test time.

**Group A (Teacher / Analysis):** Uses provided annotations **only** for training
and analysis. Not part of the final deployed system.

---

## Pipeline Steps

| Step | Description | Status |
|------|-------------|--------|
| **1** | Data & Split (no models) | Done |
| **2** | Group B: Video-only CV extraction | Done |
| **3** | Group A: Teacher label generation | Done |
| **4** | CNN training (pixels to press/no-press) | Done |
| **5** | Temporal refinement (BiLSTM/GRU) | Done |

---
## 0. Setup

In [ ]:
import os, sys, subprocess

IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in dir() else False

if IN_COLAB:
    REPO_URL  = 'https://github.com/esnylmz/computer-vision.git'
    BRANCH    = 'besn5'
    if not os.path.exists('computer-vision'):
        subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL], check=True)
    os.chdir('computer-vision')
    subprocess.run(['git', 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', 'checkout', BRANCH], check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
    # mediapipe-numpy2 provides mp.solutions (removed in mediapipe 0.10.31+)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'mediapipe-numpy2', 'opencv-python', 'tqdm', 'requests'], check=True)
    print('\nColab ready')
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)
    print('Local ready')

import numpy as np
import pandas as pd
import cv2
import json
import warnings
from pathlib import Path

warnings.filterwarnings('ignore', category=UserWarning)

print(f'NumPy  : {np.__version__}')
print(f'OpenCV : {cv2.__version__}')
print(f'Pandas : {pd.__version__}')

---
## 1. STEP 1 — Data & Split (no models)

**Goal:** Sample N videos from the TRAIN split, create a video-level train/test split,
download with caching, and produce a manifest file.

**Smoke test defaults:** N=3, clip_duration=20s, frame_step=10

In [ ]:
# ── Configurable parameters (change these for full run) ──────────
N              = 3       # number of videos  (full: 60)
CLIP_DURATION  = 20      # seconds           (full: 120)
FRAME_STEP     = 10      # sample every Nth  (full: 5)
EPOCHS         = 1       # training epochs   (full: 10+)
SEED           = 42
CACHE_DIR      = '/content/data/cache' if IN_COLAB else './data/cache'
OUTPUT_DIR     = '/content/outputs'    if IN_COLAB else './outputs'

print(f'N             = {N}')
print(f'CLIP_DURATION = {CLIP_DURATION}s')
print(f'FRAME_STEP    = {FRAME_STEP}')
print(f'EPOCHS        = {EPOCHS}')
print(f'CACHE_DIR     = {CACHE_DIR}')
print(f'OUTPUT_DIR    = {OUTPUT_DIR}')

In [ ]:
from src.data import sample_and_split, download_manifest_videos, save_manifest

# 1a) Sample N videos from TRAIN split, create video-level split
manifest, dataset = sample_and_split(
    N=N,
    clip_duration=CLIP_DURATION,
    frame_step=FRAME_STEP,
    cache_dir=CACHE_DIR,
    seed=SEED,
)

In [ ]:
# 1b) Download / cache videos and verify FPS from file headers
local_paths = download_manifest_videos(manifest, dataset, verify_fps=True)
manifest['local_video_path'] = manifest['video_id'].map(local_paths)

In [ ]:
# 1c) Save manifest to disk
out_dir = Path(OUTPUT_DIR) / 'smoke_test'
out_dir.mkdir(parents=True, exist_ok=True)
manifest_path = out_dir / 'manifest.json'
save_manifest(manifest, str(manifest_path))

### Manifest Preview

In [ ]:
# 1d) Preview manifest
preview_cols = ['video_id', 'split', 'clip_start', 'clip_duration',
                'fps', 'frame_step', 'composer', 'piece']
print('--- Manifest Preview ---')
print(manifest[preview_cols].to_string(index=False))
print()

n_train = int((manifest['split'] == 'train').sum())
n_test  = int((manifest['split'] == 'test').sum())
print(f'Train videos : {n_train}')
print(f'Test videos  : {n_test}')
print(f'Manifest     : {manifest_path}')
print()

# Explicit leakage assertion (redundant safety check)
train_ids = set(manifest.loc[manifest['split'] == 'train', 'video_id'])
test_ids  = set(manifest.loc[manifest['split'] == 'test',  'video_id'])
assert train_ids.isdisjoint(test_ids), \
    f'LEAKAGE: {train_ids & test_ids}'
print('Leakage assertion PASSED: no video_id in both train and test.')

In [ ]:
# 1e) Show full manifest as table
manifest

**STEP 1 COMPLETE.**

---
## 2. STEP 2 — Group B: Video-Only CV Extraction

**Goal:** For every video in the manifest:
1. Extract fingertip (x, y) + confidence per frame using **MediaPipe Hands** (video-only, no JSON).
2. Compute **homography** from metadata corner points and warp fingertip coords into normalised keyboard space.
3. Save **at least 10 frames per video** with fingertip overlay + rectified keyboard ROI.

No CNN, no BiLSTM yet.

In [ ]:
from src.mediapipe_extract import extract_landmarks_from_video
from src.homography import (
    parse_corners, corners_valid,
    compute_keyboard_homography, add_keyboard_coords_to_landmarks,
)
from src.viz import save_sanity_check_frames

import matplotlib.pyplot as plt

sub_dir = 'smoke_test'
viz_dir = Path(OUTPUT_DIR) / sub_dir / 'viz'
lm_dir  = Path(OUTPUT_DIR) / sub_dir / 'landmarks'
lm_dir.mkdir(parents=True, exist_ok=True)

all_landmarks = {}  # video_id -> DataFrame

for row_idx, row in manifest.iterrows():
    vid_id     = row['video_id']
    video_path = row['local_video_path']
    fps        = row['fps']
    corners    = parse_corners(row['keyboard_corners'])

    print(f'\n{"="*50}')
    print(f'[{vid_id}]')

    # ── 2a) Hand landmark extraction (video-only) ────────────
    landmarks_df, vinfo = extract_landmarks_from_video(
        video_path=video_path,
        clip_duration=CLIP_DURATION,
        frame_step=FRAME_STEP,
        fps=fps,
    )
    n_det = landmarks_df['frame_idx'].nunique() if not landmarks_df.empty else 0
    print(f'  Landmarks : {len(landmarks_df)} detections in '
          f'{n_det}/{vinfo["n_processed"]} frames '
          f'({vinfo["n_with_hands"]} had hands)')

    # ── 2b) Keyboard rectification ───────────────────────────
    if not corners_valid(corners):
        print('  WARNING: invalid keyboard corners — skipping homography')
        H, dst_size = None, None
    else:
        landmarks_df, H = add_keyboard_coords_to_landmarks(landmarks_df, corners)
        _, dst_size = compute_keyboard_homography(corners)
        print('  Homography : computed, keyboard coords added')

    # ── 2c) Visualisation sanity checks ──────────────────────
    if H is not None:
        save_sanity_check_frames(
            video_path=video_path,
            landmarks_df=landmarks_df,
            H=H, dst_size=dst_size,
            corners=corners,
            output_dir=str(viz_dir),
            video_id=vid_id,
            n_frames=10,
        )

    # ── 2d) Save per-video landmarks CSV ─────────────────────
    lm_path = lm_dir / f'{vid_id}_landmarks.csv'
    landmarks_df.to_csv(lm_path, index=False)
    print(f'  Saved      : {lm_path}')

    all_landmarks[vid_id] = landmarks_df

total_det = sum(len(df) for df in all_landmarks.values())
print(f'\n{"="*50}')
print(f'STEP 2 COMPLETE — {total_det} total fingertip detections '
      f'across {len(all_landmarks)} videos.')
print(f'  Visualisations : {viz_dir}')
print(f'  Landmarks      : {lm_dir}')

### 2e) Inspect landmark data

In [ ]:
# Show a sample of the landmark data for the first video
first_vid = list(all_landmarks.keys())[0]
sample_df = all_landmarks[first_vid]
print(f'Landmark columns: {list(sample_df.columns)}')
print(f'Shape: {sample_df.shape}')
print()
sample_df.head(15)

### 2f) Display saved overlay & rectified frames

In [ ]:
# Display a selection of saved visualisation frames
for vid_id in all_landmarks:
    overlay_dir  = viz_dir / vid_id / 'overlay'
    rect_dir     = viz_dir / vid_id / 'rectified'

    overlay_files  = sorted(overlay_dir.glob('*.jpg'))[:4]
    rect_files     = sorted(rect_dir.glob('*.jpg'))[:4]

    if not overlay_files:
        print(f'{vid_id}: no overlay frames found')
        continue

    n = len(overlay_files)
    fig, axes = plt.subplots(2, n, figsize=(5 * n, 8))
    if n == 1:
        axes = axes.reshape(2, 1)

    for i, (ov, rc) in enumerate(zip(overlay_files, rect_files)):
        img_ov = cv2.cvtColor(cv2.imread(str(ov)), cv2.COLOR_BGR2RGB)
        img_rc = cv2.cvtColor(cv2.imread(str(rc)), cv2.COLOR_BGR2RGB)
        axes[0, i].imshow(img_ov)
        axes[0, i].set_title(ov.stem, fontsize=9)
        axes[0, i].axis('off')
        axes[1, i].imshow(img_rc)
        axes[1, i].set_title('rectified', fontsize=9)
        axes[1, i].axis('off')

    fig.suptitle(f'{vid_id} — overlay (top) / rectified keyboard (bottom)', fontsize=12)
    plt.tight_layout()
    plt.show()

### 2g) Keyboard-space fingertip distribution

In [ ]:
# Scatter plot: all fingertip detections in normalised keyboard space
fig, axes = plt.subplots(1, len(all_landmarks), figsize=(7 * len(all_landmarks), 3))
if len(all_landmarks) == 1:
    axes = [axes]

colors = {'thumb': 'red', 'index': 'green', 'middle': 'blue',
          'ring': 'orange', 'pinky': 'purple'}

for ax, (vid_id, df) in zip(axes, all_landmarks.items()):
    if 'x_kbd' not in df.columns or df.empty:
        ax.set_title(f'{vid_id} (no kbd coords)')
        continue
    for fname, grp in df.groupby('finger_name'):
        ax.scatter(grp['x_kbd'], grp['y_kbd'], s=4, alpha=0.5,
                   c=colors.get(fname, 'gray'), label=fname)
    ax.set_xlim(-0.1, 1.1)
    ax.set_ylim(1.2, -0.2)  # invert y so top of keyboard is up
    ax.set_xlabel('x_kbd (left → right)')
    ax.set_ylabel('y_kbd (front → back)')
    ax.set_title(vid_id)
    ax.legend(fontsize=7, markerscale=3)
    ax.set_aspect('equal')

plt.suptitle('Fingertip positions in normalised keyboard space', fontsize=13)
plt.tight_layout()
plt.show()

**STEP 2 COMPLETE.**

---
## 3. STEP 3 — Group A: Teacher Labels (analysis only)

**Goal:** Generate frame-level press / no-press labels using MIDI timing + keyboard
proximity matching.  These teacher labels are used **only for training** — never at
inference time.

1. Load TSV annotations (MIDI onset/offset/pitch).
2. For each Group B fingertip: if a MIDI note is active **and** the fingertip is
   near that note's keyboard position -> press = 1.
3. Apply Gaussian temporal smoothing.
4. Timeline plot: raw vs smoothed labels.

In [ ]:
from src.teacher_labels import generate_teacher_labels_for_video, plot_teacher_timeline

labeled = {}  # video_id -> labeled DataFrame

for _, row in manifest.iterrows():
    vid_id = row['video_id']
    print(f'\n[{vid_id}]')

    lm_df = all_landmarks.get(vid_id)
    if lm_df is None or lm_df.empty:
        print('  No landmarks — skipping')
        continue

    # Download / load TSV annotations
    sample = dataset.get_sample_by_id(vid_id)
    if sample is not None:
        tsv_df = dataset.load_tsv_annotations(sample)
    else:
        from src.data import PianoVAMDataset as _D
        tsv_url = _D.BASE_URL + f'TSV/{vid_id}.tsv'
        local = dataset.download_file(tsv_url)
        tsv_df = pd.read_csv(local, sep='\t',
                             names=['onset','key_offset','frame_offset','note','velocity'],
                             header=None, comment='#')

    labeled_df = generate_teacher_labels_for_video(
        lm_df, tsv_df,
        fps=row['fps'],
        clip_duration=CLIP_DURATION,
    )

    # Save
    out_path = lm_dir / f'{vid_id}_labeled.csv'
    labeled_df.to_csv(out_path, index=False)
    print(f'  Saved -> {out_path}')
    labeled[vid_id] = labeled_df

total = sum(len(d) for d in labeled.values())
n_press = sum(int(d['press_raw'].sum()) for d in labeled.values())
print(f'\nSTEP 3: {total} labelled samples, {n_press} press events.')

In [ ]:
# 3b) Timeline plot: raw vs smoothed teacher labels
fig, axes = plt.subplots(len(labeled), 1, figsize=(14, 3 * len(labeled)), squeeze=False)
for i, (vid_id, ldf) in enumerate(labeled.items()):
    plot_teacher_timeline(ldf, vid_id, hand='right', finger_name='index', ax=axes[i, 0])
plt.tight_layout()
plt.show()
print('STEP 3 COMPLETE.')

---
## 4. STEP 4 — CNN Training (pixels to press/no-press)

**Goal:** Train a CNN on fingertip-centred image crops.

- **Input:** 64x64 pixel crops centred on each fingertip. The CNN receives **only pixels** — no landmark coordinates.
- **Labels:** Teacher labels from Step 3 (used only during training).
- **Augmentation:** Random brightness, flip, noise.
- **Evaluation:** On TEST videos only — precision, recall, F1, ROC-AUC.

In [ ]:
import torch
from src.crops import extract_crops_for_video, PressCropDataset
from src.cnn import train_cnn, predict_cnn
from src.eval import evaluate_predictions, save_eval_plots

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# 4a) Extract crops + labels per split
train_crops, train_labels = [], []
test_crops, test_labels = [], []

for _, row in manifest.iterrows():
    vid_id = row['video_id']
    split  = row['split']
    ldf    = labeled.get(vid_id)
    if ldf is None or ldf.empty or 'press_smooth' not in ldf.columns:
        print(f'  [{vid_id}] no teacher labels — skipping')
        continue

    print(f'  [{vid_id}] extracting crops ({split}) ...')
    crops, idxs = extract_crops_for_video(row['local_video_path'], ldf, crop_size=64)
    labs = ldf.loc[idxs, 'press_smooth'].values.tolist()

    if split == 'train':
        train_crops.extend(crops)
        train_labels.extend(labs)
    else:
        test_crops.extend(crops)
        test_labels.extend(labs)

print(f'\nTrain crops: {len(train_crops)}  (pos={sum(1 for l in train_labels if l > 0.5)})')
print(f'Test crops : {len(test_crops)}  (pos={sum(1 for l in test_labels if l > 0.5)})')

In [ ]:
# 4b) Train CNN
n_pos = max(sum(1 for l in train_labels if l > 0.5), 1)
n_neg = max(len(train_labels) - n_pos, 1)
pos_weight = n_neg / n_pos
print(f'pos_weight: {pos_weight:.1f}')

train_ds = PressCropDataset(train_crops, train_labels)
cnn_model, losses = train_cnn(
    train_ds,
    epochs=EPOCHS,
    batch_size=32,
    lr=1e-3,
    device=DEVICE,
    pos_weight=pos_weight,
)

# Save model
model_path = out_dir / 'pressnet.pt'
torch.save(cnn_model.state_dict(), model_path)
print(f'Model saved -> {model_path}')

In [ ]:
# 4c) Evaluate on TEST videos
if test_crops:
    test_ds = PressCropDataset(test_crops, test_labels)
    cnn_probs_test = predict_cnn(cnn_model, test_ds, device=DEVICE)

    y_true = np.array(test_labels)
    cnn_metrics = evaluate_predictions(y_true, cnn_probs_test, label='CNN')

    eval_dir = out_dir / 'eval'
    save_eval_plots(y_true, cnn_probs_test, str(eval_dir), label='CNN')

    # Show plots inline
    for fname in ['confusion_matrix_CNN.png', 'roc_curve_CNN.png']:
        p = eval_dir / fname
        if p.exists():
            img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
            fig, ax = plt.subplots(figsize=(5, 4))
            ax.imshow(img); ax.axis('off'); ax.set_title(fname)
            plt.show()
else:
    print('No test data — skipping evaluation')

In [ ]:
# 4d) Write CNN predictions back into labelled DataFrames (needed for Step 5)
for _, row in manifest.iterrows():
    vid_id = row['video_id']
    ldf = labeled.get(vid_id)
    if ldf is None or ldf.empty or 'press_smooth' not in ldf.columns:
        continue
    print(f'  [{vid_id}] predicting ...')
    crops_v, idxs_v = extract_crops_for_video(row['local_video_path'], ldf, crop_size=64)
    if not crops_v:
        continue
    ds_v = PressCropDataset(crops_v, [0.0] * len(crops_v))
    probs_v = predict_cnn(cnn_model, ds_v, device=DEVICE)
    ldf.loc[idxs_v, 'press_prob'] = probs_v
    labeled[vid_id] = ldf
    ldf.to_csv(lm_dir / f'{vid_id}_labeled.csv', index=False)

print('STEP 4 COMPLETE.')

---
## 5. STEP 5 — Temporal Refinement (BiLSTM)

**Goal:** Refine CNN predictions temporally.

1. Build per-fingertip sequences: `[press_prob, dx, dy, speed]`.
2. Train a BiLSTM on TRAIN data.
3. Compare CNN-only vs CNN+BiLSTM on TEST data.
4. Event-consistency metric (isolated single-frame presses).
5. Timeline plot showing improvement.

In [ ]:
from src.bilstm import build_sequences, train_refiner, predict_refiner
from src.eval import event_consistency

# 5a) Build training sequences from TRAIN videos
train_feats, train_labs = [], []
for _, row in manifest.iterrows():
    if row['split'] != 'train':
        continue
    ldf = labeled.get(row['video_id'])
    if ldf is None or 'press_prob' not in ldf.columns:
        continue
    f, l = build_sequences(ldf, press_prob_col='press_prob')
    train_feats.extend(f)
    train_labs.extend(l)

print(f'Train sequences: {len(train_feats)}')

# Compute pos_weight for BiLSTM
n_pos_s = sum(float((l > 0.5).sum()) for l in train_labs)
n_neg_s = sum(float(len(l) - (l > 0.5).sum()) for l in train_labs)
pw_s = max(n_neg_s, 1) / max(n_pos_s, 1)
print(f'pos_weight: {pw_s:.1f}')

In [ ]:
# 5b) Train BiLSTM
if train_feats:
    bilstm, bilstm_losses = train_refiner(
        (train_feats, train_labs),
        epochs=EPOCHS,
        device=DEVICE,
        pos_weight=pw_s,
    )
    torch.save(bilstm.state_dict(), out_dir / 'bilstm_refiner.pt')
    print(f'BiLSTM saved -> {out_dir / "bilstm_refiner.pt"}')
else:
    bilstm = None
    print('No training sequences — skipping BiLSTM')

In [ ]:
# 5c) Predict on TEST + compare CNN vs CNN+BiLSTM
test_true_all, cnn_prob_all, refined_all = [], [], []

if bilstm is not None:
    for _, row in manifest.iterrows():
        if row['split'] != 'test':
            continue
        ldf = labeled.get(row['video_id'])
        if ldf is None or 'press_prob' not in ldf.columns:
            continue

        refined = predict_refiner(bilstm, ldf, press_prob_col='press_prob', device=DEVICE)
        ldf = ldf.copy()
        ldf['press_prob_refined'] = refined.values
        ldf.to_csv(lm_dir / f"{row['video_id']}_labeled.csv", index=False)
        labeled[row['video_id']] = ldf

        valid = ldf.dropna(subset=['press_prob', 'press_prob_refined', 'press_smooth'])
        if not valid.empty:
            test_true_all.append(valid['press_smooth'].values)
            cnn_prob_all.append(valid['press_prob'].values)
            refined_all.append(valid['press_prob_refined'].values)

if test_true_all:
    y_true_all  = np.concatenate(test_true_all)
    y_cnn_all   = np.concatenate(cnn_prob_all)
    y_ref_all   = np.concatenate(refined_all)

    print('--- CNN only ---')
    m_cnn = evaluate_predictions(y_true_all, y_cnn_all, label='CNN')
    print('--- CNN + BiLSTM ---')
    m_ref = evaluate_predictions(y_true_all, y_ref_all, label='CNN+BiLSTM')

    save_eval_plots(y_true_all, y_ref_all, str(out_dir / 'eval'), label='CNN+BiLSTM')

    # Event consistency
    ec_cnn = event_consistency((y_cnn_all > 0.5).astype(int))
    ec_ref = event_consistency((y_ref_all > 0.5).astype(int))
    print(f'\nEvent consistency (lower isolation = better):')
    print(f'  CNN only   : {ec_cnn}')
    print(f'  CNN+BiLSTM : {ec_ref}')
else:
    print('No test predictions — skipping comparison')

In [ ]:
# 5d) Timeline plot: teacher vs CNN vs CNN+BiLSTM
for _, row in manifest.iterrows():
    if row['split'] != 'test':
        continue
    ldf = labeled.get(row['video_id'])
    if ldf is None or 'press_prob_refined' not in ldf.columns:
        continue

    # Pick a fingertip with data
    for fname in ['index', 'middle', 'thumb']:
        sub = ldf[(ldf['hand'] == 'right') & (ldf['finger_name'] == fname)].sort_values('frame_idx')
        if len(sub) > 5:
            break
    else:
        continue

    fig, ax = plt.subplots(figsize=(14, 3))
    ax.fill_between(sub['time_sec'], 0, sub['press_smooth'],
                    alpha=0.2, color='gray', label='teacher')
    ax.plot(sub['time_sec'], sub['press_prob'],
            linewidth=1, color='orange', label='CNN')
    ax.plot(sub['time_sec'], sub['press_prob_refined'],
            linewidth=1.5, color='blue', label='CNN+BiLSTM')
    ax.set_ylim(-0.05, 1.15)
    ax.set_xlabel('time (s)')
    ax.set_ylabel('press prob')
    ax.set_title(f'Temporal refinement — {row["video_id"]} right {fname}')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
    break  # one plot is enough

print('STEP 5 COMPLETE.')

---
## Summary

| Component | Method | Used at inference? |
|-----------|--------|-------------------|
| Hand landmarks | MediaPipe Hands on raw video | Yes (Group B) |
| Keyboard rectification | Homography from metadata corners | Yes (Group B) |
| Teacher labels | MIDI timing + keyboard proximity | No (training only) |
| Press classifier | CNN on 64x64 fingertip crops | Yes (Group B) |
| Temporal refinement | BiLSTM over [prob, dx, dy, speed] | Yes (Group B) |

**Group B is the deployable system.** It runs on raw video only.
**Group A labels** were used only for training.